In [1]:
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()

print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [4]:
print(f"dataset length:")
print(f"{len(text)} chars")
print(f"{len(text.split(' '))} words")

dataset length:
1115394 chars
169893 words


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


Tokenizer

In [6]:
stoi = {char:i for i,char in enumerate(chars)}
itos = {i:char for char,i in stoi.items()}
encode = lambda s: [stoi[c] for c in s] # string -> list[int]
decode = lambda l: ''.join(itos[i] for i in l) # list[int] -> string

print(decode(encode("Naman")))

Naman


In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


Train and Validation Data Sets

In [8]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
print(train_data[:block_size + 1])
print(decode(train_data[:block_size + 1].tolist()))

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])
First Cit


In [10]:
x = train_data[:block_size]
y = train_data[1:block_size + 1]
print(x)
print(y)

for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"context: {context} --> target: {target}")

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])
context: tensor([18]) --> target: 47
context: tensor([18, 47]) --> target: 56
context: tensor([18, 47, 56]) --> target: 57
context: tensor([18, 47, 56, 57]) --> target: 58
context: tensor([18, 47, 56, 57, 58]) --> target: 1
context: tensor([18, 47, 56, 57, 58,  1]) --> target: 15
context: tensor([18, 47, 56, 57, 58,  1, 15]) --> target: 47
context: tensor([18, 47, 56, 57, 58,  1, 15, 47]) --> target: 58


In [11]:
batch_size = 4 # batches of blocks to process in parallel
block_size = 8 # size of blocks to train on

def get_batch(split):
    match split:
        case 'train':
            data = train_data
        case 'validation':
            data = val_data
    
    indices = torch.randint(0, len(data) - block_size, size=(batch_size, ))
    
    x = torch.stack([data[i:i + block_size] for i in indices])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in indices])
    
    return x, y

x, y = get_batch('train')
print("inputs:")
print(x)
print("targets:")
print(y)
    

inputs:
tensor([[56,  2,  1, 61, 46, 63,  6,  1],
        [43, 60, 43, 56,  1, 61, 39, 57],
        [58, 53,  1, 54, 43, 39, 41, 43],
        [ 6,  1, 61, 46, 47, 57, 54, 43]])
targets:
tensor([[ 2,  1, 61, 46, 63,  6,  1, 57],
        [60, 43, 56,  1, 61, 39, 57,  1],
        [53,  1, 54, 43, 39, 41, 43,  6],
        [ 1, 61, 46, 47, 57, 54, 43, 56]])


In [16]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, index, targets=None):
        # returns (batch, time {which char of the batch}, channel {vocab_size})
        logits = self.token_embedding_table(index)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(index)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocab_size)
logits, loss = model(x, y)
print(logits.shape)
print(loss)

index = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(index, max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(5.0104, grad_fn=<NllLossBackward0>)

$ ktrz $k&dY3uAfrpFnO;$I3-ZCL ;hOq
cjjc3uaGy x.vlNqV!!pHW,eZCss:JtawGMzLJej;hkGchDXSDD3NRMzEDrF.VGP3


In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [30]:
batch_size = 32

for steps in range(10000):
    x, y = get_batch('train')
    logits, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

2.390747308731079


In [31]:
print(decode(model.generate(index, max_new_tokens=100)[0].tolist()))


Wes toks baists D y.
MBor nyo towefise bar y whtifave trisod us?
WAnot nave m f ie wng sen, prond f 
